## Импорт библиотек и настройки

In [49]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, List, Dict, Any
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_STATE = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" if (ROOT / "data").exists() else ROOT

print("✅ Окружение настроено")

✅ Окружение настроено


In [50]:
print(f"Root: {ROOT}")

Root: /home/yamshchikov/ML_practice/Torchvision_core_project/Shift_ML_Tech_Task_2/notebooks


## Загрузка данных

In [51]:
print("📥 Загрузка данных...")
train = pd.read_csv(DATA_DIR / "../data/train.csv")
test = pd.read_csv(DATA_DIR / "../data/test.csv")
bureau = pd.read_csv(DATA_DIR / "../data/bureau.csv")
transactions = pd.read_csv(DATA_DIR / "../data/transactions.csv")
previous_loans = pd.read_csv(DATA_DIR / "../data/previous_loans.csv")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Bureau: {bureau.shape} | Transactions: {transactions.shape} | Previous Loans: {previous_loans.shape}")

# Базовый EDA: баланс классов
target_rate = train['target'].mean()
print(f"📊 Доля дефолтов (target=1) в train: {target_rate:.2%}")

📥 Загрузка данных...
Train: (6488, 25) | Test: (2520, 24)
Bureau: (24689, 8) | Transactions: (353300, 6) | Previous Loans: (12762, 7)
📊 Доля дефолтов (target=1) в train: 34.22%


## Агрегация реляционных данных (1-ко-многим)

In [52]:
def aggregate_relational_data(train_df, test_df, bureau_df, prev_loans_df, tx_df):
    print("🔄 Начало агрегации реляционных данных...")
    
    # 1. Bureau
    bureau_agg = bureau_df.groupby('client_id').agg(
        bureau_accounts_count=('bureau_account_id', 'count'),
        total_credit_limit=('credit_limit', 'sum'),
        total_current_balance=('current_balance', 'sum'),
        max_dpd_last_12m=('max_dpd_last_12m', 'max'),
        active_accounts_count=('bureau_status', lambda x: (x == 'active').sum()),
    ).reset_index()
    bureau_agg['credit_utilization_ratio'] = bureau_agg['total_current_balance'] / (bureau_agg['total_credit_limit'] + 1e-5)
    bureau_agg['severe_overdue_flag'] = (bureau_agg['max_dpd_last_12m'].fillna(0) >= 30).astype(int)
    
    # 2. Previous Loans
    prev_agg = prev_loans_df.groupby('client_id').agg(
        prev_loans_count=('previous_loan_id', 'count'),
        total_prev_amount=('previous_amount', 'sum'),
        max_overdue_days=('max_overdue_days', 'max'),
        was_overdue_count=('was_overdue', 'sum'),
        closed_days_ago=('closed_days_ago', 'min'),
    ).reset_index()
    prev_agg['was_overdue_ratio'] = prev_agg['was_overdue_count'] / (prev_agg['prev_loans_count'] + 1e-5)
    prev_agg['recent_overdue_flag'] = (
        (prev_agg['was_overdue_count'] > 0) & 
        (prev_agg['closed_days_ago'].fillna(9999) < 365)
    ).astype(int)
    
    # 3. Transactions
    tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'], errors='coerce')
    ref_date = pd.to_datetime('2025-12-31')
    tx_df['days_since_tx'] = (ref_date - tx_df['transaction_date']).dt.days
    
    tx_agg = tx_df.groupby('client_id').agg(
        tx_count=('transaction_date', 'count'),
        net_tx_amount=('amount', 'sum'),
        total_tx_turnover=('amount', lambda x: x.abs().sum()),
        days_since_last_tx=('days_since_tx', 'min')
    ).reset_index()
    
    # Азартные игры (gambling)
    gambling_df = tx_df[tx_df['transaction_category'] == 'gambling']
    gambling_agg = gambling_df.groupby('client_id').agg(
        gambling_count=('transaction_category', 'count'),
        gambling_amount=('amount', lambda x: x.abs().sum()),
        gambling_max_risk=('merchant_risk_level', 'max')
    ).reset_index()
    
    # Зарплаты (salary)
    salary_df = tx_df[tx_df['transaction_category'] == 'salary']
    salary_agg = salary_df.groupby('client_id').agg(
        salary_count=('transaction_category', 'count'),
        avg_salary=('amount', 'mean'),
        days_since_last_salary=('days_since_tx', 'min')
    ).reset_index()
    
    # Высокий риск (merchant_risk_level >= 4)
    high_risk_df = tx_df[tx_df['merchant_risk_level'] >= 4]
    high_risk_agg = high_risk_df.groupby('client_id').agg(
        high_risk_count=('transaction_category', 'count')
    ).reset_index()

    # Объединяем все транзакционные агрегации
    tx_agg = tx_agg.merge(gambling_agg, on='client_id', how='left')
    tx_agg = tx_agg.merge(salary_agg, on='client_id', how='left')
    tx_agg = tx_agg.merge(high_risk_agg, on='client_id', how='left')
    
    # Заполняем пропуски нулями
    fill_zero_cols = ['gambling_count', 'gambling_amount', 'salary_count', 'high_risk_count',
                      'gambling_max_risk', 'avg_salary', 'days_since_last_salary']
    for col in fill_zero_cols:
        if col in tx_agg.columns:
            tx_agg[col] = tx_agg[col].fillna(0)
            
    tx_agg['gambling_to_turnover_ratio'] = tx_agg['gambling_amount'] / (tx_agg['total_tx_turnover'] + 1e-5)
    
    # 4. Слияние
    result_dfs = []
    for df in [train_df, test_df]:
        df_merged = df.copy()
        df_merged = df_merged.merge(bureau_agg, on='client_id', how='left')
        df_merged = df_merged.merge(prev_agg, on='client_id', how='left')
        df_merged = df_merged.merge(tx_agg, on='client_id', how='left')
        result_dfs.append(df_merged)
        
    print("✅ Агрегация завершена (с расширенными транзакциями и флагами).")
    return result_dfs[0], result_dfs[1]

train_agg, test_agg = aggregate_relational_data(train, test, bureau, previous_loans, transactions)

🔄 Начало агрегации реляционных данных...


✅ Агрегация завершена (с расширенными транзакциями и флагами).


## Продвинутый Feature Engineering (Адаптированный)

In [ ]:
def create_new_features(df):
    df = df.copy()
    
    # 1. Безопасная обработка дохода для расчетов
    df['monthly_income_safe'] = df['monthly_income'].fillna(df['monthly_income'].median())
    
    # 2. Отношения и нагрузки
    eps = 1e-6
    df['loan_to_income_ratio'] = df['loan_amount'] / (df['monthly_income_safe'] * 12 + eps)
    df['estimated_monthly_payment'] = df['loan_amount'] / (df['loan_term_months'].fillna(12) + eps)
    df['payment_to_income_ratio'] = df['estimated_monthly_payment'] / (df['monthly_income_safe'] + eps)
    
    # 3. Логарифмирование
    df['log_loan_amount'] = np.log1p(df['loan_amount'])
    df['log_monthly_income'] = np.log1p(df['monthly_income_safe'])
    df['log_total_credit_limit'] = np.log1p(df['total_credit_limit'].fillna(0) + 1)
    
    # 4. Риски из бюро и истории
    df['has_overdue_history'] = (df['max_dpd_last_12m'].fillna(0) > 0).astype(int)
    df['bureau_debt_ratio'] = df['total_current_balance'].fillna(0) / (df['total_credit_limit'].fillna(1) + eps)
    
    # 5. Активность транзакций
    df['tx_turnover_to_income'] = df['total_tx_turnover'].fillna(0) / (df['monthly_income_safe'] * 12 + eps)
    
    # 6. Флаги занятости
    df['is_high_risk_employment'] = df['employment_type'].isin(
        ['unemployed', 'contractor', 'self_employed', 'business']
    ).astype(int)
    df['is_short_term_loan'] = (df['loan_term_months'].fillna(12) <= 12).astype(int)
    
    # 7. Возраст клиента как риск
    df['is_young'] = (df['age'].fillna(35) < 25).astype(int)
    df['is_senior'] = (df['age'].fillna(35) > 60).astype(int)
    
    # 8. Стаж на работе как стабильность
    df['is_new_employee'] = (df['months_at_job'].fillna(0) < 12).astype(int)
    
    # 9. Соотношение запрошенной суммы к доходу (с учётом dependents)
    df['dependents'] = df['dependents'].fillna(0)
    df['income_per_dependent'] = df['monthly_income_safe'] / (df['dependents'] + 1)
    
    # 10. siberia_northern_score
    if 'siberia_northern_score' in df.columns:
        df['log_siberia_score'] = np.log1p(df['siberia_northern_score'].fillna(0))
    
    return df

def add_kfold_target_encoding(train_df, test_df, cat_cols, target_col='target', n_splits=5):
    """Безопасный Target Encoding с использованием K-Fold (без утечки данных)"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    global_mean = train_df[target_col].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    for col in cat_cols:
        train_df[f'{col}_target_enc'] = np.nan
        test_df[f'{col}_target_enc'] = global_mean 
        
        for train_idx, val_idx in kf.split(train_df):
            X_train_fold = train_df.iloc[train_idx]
            X_val_fold = train_df.iloc[val_idx]
            
            enc_map = X_train_fold.groupby(col)[target_col].mean()
            train_df.loc[val_idx, f'{col}_target_enc'] = X_val_fold[col].map(enc_map)
            
        train_df[f'{col}_target_enc'] = train_df[f'{col}_target_enc'].fillna(global_mean)
        
        test_enc_map = train_df.groupby(col)[target_col].mean()
        test_df[f'{col}_target_enc'] = test_df[col].map(test_enc_map).fillna(global_mean)
        
    return train_df, test_df

cat_cols_to_encode = ['employment_type', 'region', 'education', 'requested_product', 'channel']
train_fe, test_fe = add_kfold_target_encoding(train_agg, test_agg, cat_cols_to_encode, target_col='target')

train_fe = create_new_features(train_fe)
test_fe = create_new_features(test_fe)
print("✅ Улучшенный Feature Engineering завершен")

✅ Улучшенный Feature Engineering завершен


## Умная предобработка

In [ ]:
def advanced_preprocessing_catboost(train_df, test_df, target_col='target'):
    print("="*50)
    print("НАЧАЛО ПРЕДОБРАБОТКИ (CatBoost Native)")
    print("="*50)
    
    leakage_cols = [
        'days_until_first_overdue',   # Прямая утечка: 999/1001 для нормы, конкретные дни для дефолта
        'internal_decision_code',     # Прямая утечка: late_flag/collections = дефолт по определению
        'post_loan_collection_score', # Post-factum скор, рассчитанный после события
    ]
    
    cols_to_drop = [c for c in leakage_cols if c in train_df.columns]
    print(f"🔴 Удалены признаки-утечки: {cols_to_drop}")

    id_cols = ['application_id', 'client_id', 'hash_id']
    drop_all = cols_to_drop + id_cols + [target_col]

    X_train = train_df.drop(columns=[c for c in drop_all if c in train_df.columns], errors='ignore')
    X_test = test_df.drop(columns=[c for c in id_cols if c in test_df.columns], errors='ignore')
    y_train = train_df[target_col]
    test_ids = test_df['application_id']

    X_test = X_test.drop(columns=[c for c in leakage_cols if c in X_test.columns], errors='ignore')
    
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    
    print(f"Числовых: {len(num_cols)}, Категориальных: {len(cat_cols)}")
    
    # 1. Числовые: заполняем медианой из TRAIN
    for col in num_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        
    # 2. Категориальные: заполняем 'MISSING' и оставляем как строки
    for col in cat_cols:
        X_train[col] = X_train[col].fillna('MISSING').astype(str)
        X_test[col] = X_test[col].fillna('MISSING').astype(str)
        
    # 3. Выравнивание колонок
    missing_in_test = set(X_train.columns) - set(X_test.columns)
    for col in missing_in_test:
        X_test[col] = 'MISSING' if col in cat_cols else 0
        
    extra_in_test = set(X_test.columns) - set(X_train.columns)
    X_test = X_test.drop(columns=list(extra_in_test), errors='ignore')
    X_test = X_test[X_train.columns]
    
    print("✅ Предобработка завершена.")
    return X_train, X_test, y_train, test_ids, cat_cols, num_cols

X_train, X_test, y_train, test_ids, cat_cols, num_cols = advanced_preprocessing_catboost(train_fe, test_fe)

НАЧАЛО ПРЕДОБРАБОТКИ (CatBoost Native)
🔴 Удалены признаки-утечки: ['days_until_first_overdue', 'internal_decision_code', 'post_loan_collection_score']
Числовых: 59, Категориальных: 7
✅ Предобработка завершена.


## Разделение на Train/Validation и Обучение Моделей

In [55]:
print("🔍 Фильтрация признаков...")

# 1. Удаление сильно скоррелированных признаков (порог 0.90 для деревьев)
corr_matrix = X_train[num_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop_corr = [column for column in upper.columns if any(upper[column] > 0.90)]
print(f"Удалено {len(to_drop_corr)} сильно скоррелированных признаков.")

# 2. Удаление признаков с околонулевой дисперсией
variances = X_train[num_cols].var(skipna=True)
to_drop_var = variances[variances < 1e-4].index.tolist()
print(f"Удалено {len(to_drop_var)} признаков с низкой дисперсией.")

# Объединяем списки для удаления
drop_cols = list(set(to_drop_corr + to_drop_var))

# Применяем фильтрацию
X_train_filtered = X_train.drop(columns=drop_cols, errors='ignore')
X_test_filtered = X_test.drop(columns=drop_cols, errors='ignore')

# Обновляем списки колонок
final_num_cols = [c for c in num_cols if c not in drop_cols]
final_cat_cols = [c for c in cat_cols if c not in drop_cols]

print(f"✅ Осталось признаков для обучения: {X_train_filtered.shape[1]}")

🔍 Фильтрация признаков...
Удалено 6 сильно скоррелированных признаков.
Удалено 2 признаков с низкой дисперсией.
✅ Осталось признаков для обучения: 58


In [ ]:
print("Проверка на оставшиеся утечки данных...")
correlations_with_target = X_train_filtered[final_num_cols].corrwith(y_train).abs().sort_values(ascending=False)

print("\nТоп-10 признаков по корреляции с target:")
print(correlations_with_target.head(10))

suspicious = correlations_with_target[correlations_with_target > 0.9]
if len(suspicious) > 0:
    print(f"\n⚠️ ОБНАРУЖЕНЫ ПОДОЗРИТЕЛЬНЫЕ ПРИЗНАКИ (корреляция > 0.9):")
    for col, corr in suspicious.items():
        print(f"  {col}: {corr:.4f}")
    print("→ Рекомендуется удалить их и пересчитать.")
    
    # Автоматическое удаление
    X_train_filtered = X_train_filtered.drop(columns=suspicious.index.tolist(), errors='ignore')
    X_test_filtered = X_test_filtered.drop(columns=suspicious.index.tolist(), errors='ignore')
    final_num_cols = [c for c in final_num_cols if c not in suspicious.index.tolist()]
    print(f"✅ Удалено {len(suspicious)} подозрительных признаков.")
else:
    print("✅ Подозрительных признаков не обнаружено.")

🔍 Проверка на оставшиеся утечки данных...

Топ-10 признаков по корреляции с target:
max_dpd_last_12m              0.348404
severe_overdue_flag           0.292119
payment_to_income_ratio       0.280196
months_at_job                 0.238831
loan_to_income_ratio          0.236080
employment_type_target_enc    0.224460
estimated_monthly_payment     0.211477
age                           0.175226
incoming_amount               0.160331
has_overdue_history           0.154647
dtype: float64
✅ Подозрительных признаков не обнаружено.


In [ ]:
def run_stacking_ensemble(X_train, y_train, X_test, cat_features_list):
    print("🚀 Запуск 5-Fold Stacking (LGBM + CatBoost + XGBoost -> Logistic Regression)")
    print("🔧 Подготовка числовой версии данных для LightGBM и XGBoost...")
    X_train_num = X_train.copy()
    X_test_num = X_test.copy()
    
    le_dict = {}
    for col in cat_features_list:
        le = LabelEncoder()
        combined = pd.concat([X_train_num[col].astype(str), X_test_num[col].astype(str)])
        le.fit(combined)
        X_train_num[col] = le.transform(X_train_num[col].astype(str))
        X_test_num[col] = le.transform(X_test_num[col].astype(str))
        le_dict[col] = le
    
    print(f"✅ Закодировано {len(cat_features_list)} категориальных признаков")
    
    n_splits = 5
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    oof_lgb = np.zeros(len(X_train))
    oof_cb = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))
    
    test_lgb = np.zeros(len(X_test))
    test_cb = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))
    
    # Списки для хранения скоров каждого фолда
    fold_scores_lgb = []
    fold_scores_cb = []
    fold_scores_xgb = []
    
    # Инициализация моделей
    # lgb_model = lgb.LGBMClassifier(
    #     n_estimators=1500, learning_rate=0.03, max_depth=8, num_leaves=63, 
    #     subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, 
    #     random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, is_unbalance=True
    # )

    lgb_model = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.02, max_depth=7, num_leaves=63,
            subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
            min_child_samples=50, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
            is_unbalance=True
        )
    
    # cb_model = CatBoostClassifier(
    #     iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=5, 
    #     random_state=RANDOM_STATE, verbose=0, auto_class_weights='Balanced', 
    #     grow_policy="SymmetricTree", bootstrap_type="Bayesian", bagging_temperature=1.0
    # )

    cb_model = CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=6, l2_leaf_reg=5,
            random_state=RANDOM_STATE, verbose=0, auto_class_weights='Balanced',
            grow_policy="SymmetricTree", bootstrap_type="Bayesian", bagging_temperature=1.0
        )
    
    # xgb_model = xgb.XGBClassifier(
    #     n_estimators=1500, learning_rate=0.03, max_depth=6, subsample=0.8, 
    #     colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, 
    #     random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc'
    # )

    xgb_model = xgb.XGBClassifier(
            n_estimators=2000, learning_rate=0.02, max_depth=6, subsample=0.8,
            colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=-1, eval_metric='auc'
        )
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
        print(f"\n--- Fold {fold+1}/{n_splits} ---")
        
        X_tr_cat, X_val_cat = X_train.iloc[train_idx], X_train.iloc[val_idx]
        X_tr_num, X_val_num = X_train_num.iloc[train_idx], X_train_num.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # --- LightGBM ---
        lgb_model.fit(X_tr_num, y_tr, eval_set=[(X_val_num, y_val)], 
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof_lgb[val_idx] = lgb_model.predict_proba(X_val_num)[:, 1]
        test_lgb += lgb_model.predict_proba(X_test_num)[:, 1] / n_splits
        
        # --- CatBoost ---
        cb_model.fit(X_tr_cat, y_tr, eval_set=(X_val_cat, y_val), cat_features=cat_features_list, 
                     early_stopping_rounds=100, verbose=0)
        oof_cb[val_idx] = cb_model.predict_proba(X_val_cat)[:, 1]
        test_cb += cb_model.predict_proba(X_test)[:, 1] / n_splits
        
        # --- XGBoost ---
        xgb_model.fit(X_tr_num, y_tr, eval_set=[(X_val_num, y_val)], verbose=0)
        oof_xgb[val_idx] = xgb_model.predict_proba(X_val_num)[:, 1]
        test_xgb += xgb_model.predict_proba(X_test_num)[:, 1] / n_splits
        
        # Оценка на валидации фолдов
        auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
        auc_cb = roc_auc_score(y_val, oof_cb[val_idx])
        auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
        
        fold_scores_lgb.append(auc_lgb)
        fold_scores_cb.append(auc_cb)
        fold_scores_xgb.append(auc_xgb)
        
        print(f"  Fold {fold+1} Val AUC -> LGBM: {auc_lgb:.5f} | CatBoost: {auc_cb:.5f} | XGB: {auc_xgb:.5f}")

    # Расчет среднего OOF скора по всем фолдам
    mean_oof_lgb = np.mean(fold_scores_lgb)
    mean_oof_cb = np.mean(fold_scores_cb)
    mean_oof_xgb = np.mean(fold_scores_xgb)
    mean_oof_avg = (mean_oof_lgb + mean_oof_cb + mean_oof_xgb) / 3
    
    print("\n" + "="*60)
    print("📊 СВОДКА ПО КРОСС-ВАЛИДАЦИИ (Оценка для лидерборда)")
    print("="*60)
    print(f"Средний OOF AUC LightGBM:  {mean_oof_lgb:.5f} (±{np.std(fold_scores_lgb):.5f})")
    print(f"Средний OOF AUC CatBoost:   {mean_oof_cb:.5f} (±{np.std(fold_scores_cb):.5f})")
    print(f"Средний OOF AUC XGBoost:    {mean_oof_xgb:.5f} (±{np.std(fold_scores_xgb):.5f})")
    print(f"➡️  СРЕДНИЙ OOF AUC ВСЕХ МОДЕЛЕЙ: {mean_oof_avg:.5f}")
    print("="*60)

    # Мета-модель
    print("\n🧠 Обучение мета-модели (Logistic Regression) на OOF предсказаниях...")
    X_meta_train = np.column_stack((oof_lgb, oof_cb, oof_xgb))
    X_meta_test = np.column_stack((test_lgb, test_cb, test_xgb))
    
    meta_model = LogisticRegression(C=1.0, random_state=RANDOM_STATE, max_iter=1000)
    meta_model.fit(X_meta_train, y_train)
    
    final_test_preds = meta_model.predict_proba(X_meta_test)[:, 1]
    
    # Оценка мета-модели на OOF данных
    oof_meta_preds = meta_model.predict_proba(X_meta_train)[:, 1]
    stacking_oof_auc = roc_auc_score(y_train, oof_meta_preds)
    
    print(f"✅ Итоговый Stacking OOF AUC (Meta-Model): {stacking_oof_auc:.5f}")
    print(f"   (Ожидаемый скор на платформе: ~{stacking_oof_auc:.4f} ± 0.01)")
    
    return final_test_preds, stacking_oof_auc, mean_oof_avg

In [58]:
final_test_preds, stacking_oof_auc, mean_oof_avg = run_stacking_ensemble(
    X_train_filtered, y_train, X_test_filtered, final_cat_cols
)

🚀 Запуск 5-Fold Stacking (LGBM + CatBoost + XGBoost -> Logistic Regression)
🔧 Подготовка числовой версии данных для LightGBM и XGBoost...


✅ Закодировано 7 категориальных признаков

--- Fold 1/5 ---
  Fold 1 Val AUC -> LGBM: 0.84175 | CatBoost: 0.84606 | XGB: 0.82857

--- Fold 2/5 ---
  Fold 2 Val AUC -> LGBM: 0.80686 | CatBoost: 0.81579 | XGB: 0.80072

--- Fold 3/5 ---
  Fold 3 Val AUC -> LGBM: 0.81908 | CatBoost: 0.82031 | XGB: 0.81429

--- Fold 4/5 ---
  Fold 4 Val AUC -> LGBM: 0.83010 | CatBoost: 0.84268 | XGB: 0.82517

--- Fold 5/5 ---
  Fold 5 Val AUC -> LGBM: 0.83270 | CatBoost: 0.84354 | XGB: 0.82458

📊 СВОДКА ПО КРОСС-ВАЛИДАЦИИ (Оценка для лидерборда)
Средний OOF AUC LightGBM:  0.82610 (±0.01203)
Средний OOF AUC CatBoost:   0.83368 (±0.01289)
Средний OOF AUC XGBoost:    0.81866 (±0.01017)
➡️  СРЕДНИЙ OOF AUC ВСЕХ МОДЕЛЕЙ: 0.82615

🧠 Обучение мета-модели (Logistic Regression) на OOF предсказаниях...
✅ Итоговый Stacking OOF AUC (Meta-Model): 0.83345
   (Ожидаемый скор на платформе: ~0.8334 ± 0.01)


## Попытка 1

🚀 Запуск 5-Fold Stacking (LGBM + CatBoost + XGBoost -> Logistic Regression)
🔧 Подготовка числовой версии данных для LightGBM и XGBoost...
✅ Закодировано 7 категориальных признаков

--- Fold 1/5 ---
  Fold 1 Val AUC -> LGBM: 0.83681 | CatBoost: 0.84606 | XGB: 0.82454

--- Fold 2/5 ---
  Fold 2 Val AUC -> LGBM: 0.79943 | CatBoost: 0.81579 | XGB: 0.79854

--- Fold 3/5 ---
  Fold 3 Val AUC -> LGBM: 0.81920 | CatBoost: 0.82031 | XGB: 0.81263

--- Fold 4/5 ---
  Fold 4 Val AUC -> LGBM: 0.82570 | CatBoost: 0.84268 | XGB: 0.82602

--- Fold 5/5 ---
  Fold 5 Val AUC -> LGBM: 0.83171 | CatBoost: 0.84354 | XGB: 0.81947

============================================================
📊 СВОДКА ПО КРОСС-ВАЛИДАЦИИ (Оценка для лидерборда)
============================================================
Средний OOF AUC LightGBM:  0.82257 (±0.01299)
Средний OOF AUC CatBoost:   0.83368 (±0.01289)
Средний OOF AUC XGBoost:    0.81624 (±0.01001)
➡️  СРЕДНИЙ OOF AUC ВСЕХ МОДЕЛЕЙ: 0.82416
============================================================

🧠 Обучение мета-модели (Logistic Regression) на OOF предсказаниях...
✅ Итоговый Stacking OOF AUC (Meta-Model): 0.83343
   (Ожидаемый скор на платформе: ~0.8334 ± 0.01)

In [59]:
print("\n" + "="*60)
print("🎯 ФИНАЛЬНОЕ ПРЕДСКАЗАНИЕ И ГЕНЕРАЦИЯ САБМИТА (Stacking)")
print("="*60)

val_auc_proxy = stacking_oof_auc
print(f"📊 Stacking OOF ROC-AUC (ожидаемый скор на лидерборде): {val_auc_proxy:.5f}")
print("⚠️ Примечание: Истинный скор на тесте может незначительно отличаться (±0.005).")

# Проверки
assert final_test_preds.min() >= 0.0 and final_test_preds.max() <= 1.0, "❌ Вероятности вне [0, 1]!"
assert not np.isnan(final_test_preds).any(), "❌ Обнаружены NaN!"
assert len(final_test_preds) == len(test_ids), "❌ Количество предсказаний != количеству ID!"

# Формирование submission
submission = pd.DataFrame({
    "application_id": test_ids,
    "target": final_test_preds
})

submission.to_csv(ROOT / "submission.csv", index=False)
print("✅ Файл submission.csv успешно сохранен!")
display(submission.head())

print(f"\n📈 Статистика предсказаний (target):")
print(f"  Min:    {final_test_preds.min():.4f}")
print(f"  Max:    {final_test_preds.max():.4f}")
print(f"  Mean:   {final_test_preds.mean():.4f}")
print(f"  Median: {np.median(final_test_preds):.4f}")


🎯 ФИНАЛЬНОЕ ПРЕДСКАЗАНИЕ И ГЕНЕРАЦИЯ САБМИТА (Stacking)
📊 Stacking OOF ROC-AUC (ожидаемый скор на лидерборде): 0.83345
⚠️ Примечание: Истинный скор на тесте может незначительно отличаться (±0.005).
✅ Файл submission.csv успешно сохранен!


,application_id,target
0,102531,0.760331
1,107213,0.646202
2,100238,0.188777
3,104918,0.163624
4,106480,0.409478



📈 Статистика предсказаний (target):
  Min:    0.0456
  Max:    0.8968
  Mean:   0.3454
  Median: 0.2302


## Финальное предсказание на Test и создание сабмита

## Генерация requirements.txt и архива submission.zip

In [62]:
print("\n📦 Формирование архива для отправки...")

# Создаем requirements.txt
req_content = """pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
lightgbm>=4.0.0
catboost>=1.2.0
xgboost>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
shap>=0.52.0
"""
with open(ROOT / "requirements.txt", "w") as f:
    f.write(req_content)
print("✅ requirements.txt создан")

# Создаем архив submission.zip
zip_path = ROOT / "submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(ROOT / "submission.csv", "submission.csv")
    zipf.write(ROOT / "requirements.txt", "requirements.txt")
    zipf.write(ROOT / "competition.ipynb", "competition.ipynb")

print(f"✅ Архив успешно создан: {zip_path}")
print(f"💡 Ожидаемый результат на лидерборде: ~{val_auc_proxy:.4f} (цель: > 0.8277)")


📦 Формирование архива для отправки...
✅ requirements.txt создан
✅ Архив успешно создан: /home/yamshchikov/ML_practice/Torchvision_core_project/Shift_ML_Tech_Task_2/notebooks/submission.zip
💡 Ожидаемый результат на лидерборде: ~0.8334 (цель: > 0.8277)
